# Layer Pruning Analysis

Analyze the effect of pruning layers: skip connections, layer criticality,
optimal pruning order, and performance under progressive layer removal.

**References:**
- Fan et al. (2020) "Reducing Transformer Depth on Demand with Structured Dropout"
- Men et al. (2024) "ShortGPT: Layers in Large Language Models are More Redundant Than You Expect"

## Why This Matters

Layer pruning analysis identifies which layers can be removed with minimal impact on model performance. Skip analysis, progressive pruning, and criticality measurements reveal the effective depth of the model — often much less than the architectural depth.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_pruning_analysis import (
    layer_skip_analysis,
    progressive_layer_pruning,
    layer_criticality_profile,
    layer_similarity_for_pruning,
    optimal_layer_subset,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Layer skip analysis - effect of bypassing each layer
skip = layer_skip_analysis(model, tokens, metric_fn)
print(f"Most critical layer: {skip['most_critical_layer']}")
print(f"Least critical layer: {skip['least_critical_layer']}")
print(f"Mean skip effect: {skip['mean_skip_effect']:.4f}")
print(f"\nPer-layer skip effects:")
for l in range(cfg.n_layers):
    can = 'CAN SKIP' if skip['can_skip'][l] else 'critical'
    bar = '#' * int(skip['skip_effects'][l] * 20 / (max(skip['skip_effects']) + 1e-10))
    print(f"  Layer {l}: {skip['skip_effects'][l]:.4f} ({can}) {bar}")

In [ ]:
# 2. Progressive pruning - remove layers one at a time
prog = progressive_layer_pruning(model, tokens, metric_fn)
print(f"Pruning order: {prog['pruning_order']}")
print(f"Layers before 50% loss: {prog['layers_before_50pct_loss']}")
print(f"Graceful degradation: {prog['graceful_degradation']}")
print(f"\nMetric after each pruning step:")
for i, m in enumerate(prog['metrics_after_pruning']):
    label = f'prune L{prog["pruning_order"][i-1]}' if i > 0 else 'baseline'
    print(f"  Step {i} ({label}): {m:.4f}")

In [ ]:
# 3. Layer criticality profile
crit = layer_criticality_profile(model, tokens, metric_fn)
print(f"Critical layers (above mean): {crit['critical_layers']}")
print(f"Redundant layers (<10% of max): {crit['redundant_layers']}")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: attn={crit['attn_criticality'][l]:.4f}, mlp={crit['mlp_criticality'][l]:.4f}, combined={crit['combined_criticality'][l]:.4f}")

In [ ]:
# 4. Layer similarity for pruning - find redundant layers
sim = layer_similarity_for_pruning(model, tokens)
print(f"Most similar pair: {sim['most_similar_pair']}")
print(f"Most different pair: {sim['most_different_pair']}")
print(f"\nSimilarity to identity (1.0 = pure skip):")
for l in range(cfg.n_layers):
    bar = '#' * int(max(0, sim['similarity_to_identity'][l]) * 30)
    print(f"  Layer {l}: {sim['similarity_to_identity'][l]:+.4f} {bar}")
print(f"\nLayer similarity matrix:")
for i in range(cfg.n_layers):
    row = ' '.join(f"{sim['layer_similarity_matrix'][i,j]:+.3f}" for j in range(cfg.n_layers))
    print(f"  L{i}: {row}")

In [ ]:
# 5. Optimal layer subset - best layers to keep
for target in [3, 2, 1]:
    opt = optimal_layer_subset(model, tokens, metric_fn, target_layers=target)
    print(f"Keep {target} layers: {opt['kept_layers']} (prune {opt['pruned_layers']})")
    print(f"  Retention: {opt['retention_ratio']:.2%} (subset={opt['subset_metric']:.4f}, full={opt['full_metric']:.4f})")